In [1]:
import pandas as pd
from datetime import datetime
import sqlite3
import random

In [2]:
conn = sqlite3.connect("./my_database.db")

In [3]:
vendor_ids = pd.read_sql("select distinct(vendor_id) from ticket", con=conn)

In [4]:
vendor_ids

,vendor_id
0,825dbb6e-25ae-47d4-9ba9-482a7ff7bdda
1,182a7740-adb6-439c-88da-337f4a6e246b
2,1031b882-b8c6-4750-80ab-9a4767d2da65
3,f0fd72b5-d218-4ef1-9cb7-25c2b4b3d8c6
4,751607be-e0d2-4aa6-a894-eff55c0dd09a
5,8d1566fb-d1d4-4d27-a51c-44dcb03542d9
6,11c653b3-219e-4d98-9b42-7d6f683ea5c6
7,6edb3a3d-b56e-47dd-ba75-0cee5237481b
8,6843451d-5544-4d0f-ae51-7e69b35e3b73
9,4344a225-91e6-4920-9d9a-eaf15136104b


In [5]:
test_vendor = vendor_ids.iloc[1]['vendor_id']

In [6]:
test_vendor

'182a7740-adb6-439c-88da-337f4a6e246b'

In [7]:
def active_tickets(vendor_id):
    df = pd.read_sql(f"""
    select count(*) as active_tickets
    from ticket 
    where vendor_id='{vendor_id}' and status = 'in progress'
    """, con=conn)
    count = df['active_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'active tickets': count}

In [8]:
active_tickets(test_vendor)

{'vendor_id': '182a7740-adb6-439c-88da-337f4a6e246b',
 'active tickets': np.int64(262)}

In [9]:
def vendor_on_time_delivery(vendor_id):
    df = pd.read_sql(
        f"""
        select * 
        from ticket
        where vendor_id = '{vendor_id}'
        """,
        con=conn
    )
    df['on_time'] = (
        df['completed_at'].notna() &
        (df['completed_at'] <= df['due_date'])
    )

    df = df.groupby('vendor_id', as_index=False).agg(on_time_count=("on_time", "sum"),ticket_count=("ticket_id", "count"))
    df['on_time_pct'] = df['on_time_count'] / df['ticket_count'] * 100
    print(df[['vendor_id', 'on_time_count', 'ticket_count', 'on_time_pct']])
    on_time_pct = df['on_time_pct'].iloc[0].round(2)
    return {'vendor_id': vendor_id, 'on_time_pct': on_time_pct}

In [10]:
vendor_on_time_delivery(test_vendor)

                              vendor_id  on_time_count  ticket_count  \
0  182a7740-adb6-439c-88da-337f4a6e246b             44           866   

   on_time_pct  
0     5.080831  


{'vendor_id': '182a7740-adb6-439c-88da-337f4a6e246b',
 'on_time_pct': np.float64(5.08)}

In [11]:
def flagged_tickets(vendor_id):
    df = pd.read_sql(
        f"""
            select count(*) as flagged_tickets from ticket
            where vendor_id='{vendor_id}' and anomaly_flag=True
        """, con=conn
    )
    flagged_ticket_count = df['flagged_tickets'].iloc[0]
    return {'vendor_id': vendor_id, 'flagged_ticket_count': flagged_ticket_count}

In [12]:
flagged_tickets(test_vendor)

{'vendor_id': '182a7740-adb6-439c-88da-337f4a6e246b',
 'flagged_ticket_count': np.int64(406)}

## Completed Today (work in progress due to logic of sample data)

In [13]:
df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{test_vendor}' and status='completed'
        """, parse_dates=['assigned_at', 'completed_at', 'created_at', 'updated_at', 'start_time', 'due_date'], con=conn
    )

In [14]:
max(df['completed_at'])

Timestamp('2026-04-16 15:53:26')

In [15]:
def completed_tickets_today(vendor_id):
    df = pd.read_sql(
        f"""
        select *
        from ticket
        where vendor_id='{vendor_id}'
        """, con=conn
    )
    return df

In [16]:
# completed_tickets_today(test_vendor)

## Average Duration

In [17]:
def average_duration(vendor_id):
    df = pd.read_sql(
        f"""
        select * from ticket
        where vendor_id = '{vendor_id}' and status='completed'
        """,
        con=conn,
        parse_dates=['completed_at', 'start_time']
    )
    df['duration'] = df['completed_at'] - df['start_time']
    avg_duration = df['duration'].mean()
    return {'vendor_id': vendor_id, 'avg_duration': avg_duration}

In [18]:
average_duration(test_vendor)

{'vendor_id': '182a7740-adb6-439c-88da-337f4a6e246b',
 'avg_duration': Timedelta('18 days 03:57:42.142857')}